# DFU Repair-7 CPU FINAL PLAIN-EVIDENCE
CPU-only repair runner. Uses pinned plain CSV evidence; no embedded hex/Base64 recovery. Preserves 38 good trials and authorizes only the original seven fold-1 repairs.

In [ ]:
import json, urllib.request

V17_URL = "https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/ba99c6b6da8ebd5a771e0eef6424586ee0933c02/notebooks/DFU_Repair7_v1_7_CPU_Colab.ipynb"
PATCH_URL = "https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/11881ee007149b23e3664298c0d09c1f988acdd1/notebooks/repair7_evidence/repair7_cpu_plain_evidence_patch.py"

EVIDENCE_LOGITS_URL = "https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/11881ee007149b23e3664298c0d09c1f988acdd1/notebooks/repair7_evidence/mobilenetv3_large_seed2028_fold1_logits.csv"
EVIDENCE_METRIC_URL = "https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/11881ee007149b23e3664298c0d09c1f988acdd1/notebooks/repair7_evidence/mobilenetv3_large_seed2028_fold1_metric.csv"
EVIDENCE_LOGITS_SHA256 = "c5f696aa89b0436ad100911fe8d4f478b91b86b94da6d269823d51c8308b4020"
EVIDENCE_METRIC_SHA256 = "42e183909a73db5ab1e8ae1dcf09bc9ea4b7a9507fbd23594cf160066f01a766"

v17_raw = urllib.request.urlopen(V17_URL, timeout=120).read()
v17_nb = json.loads(v17_raw.decode("utf-8"))
cells = [c for c in v17_nb.get("cells", []) if c.get("cell_type") == "code"]
if len(cells) != 1:
    raise RuntimeError(f"Expected exactly one v1.7 code cell, found {len(cells)}")
code = "".join(cells[0]["source"])

patch = urllib.request.urlopen(PATCH_URL, timeout=120).read().decode("utf-8")
required_patch_markers = [
    "Plain CSV recovery patch: PASS",
    "Locked relative-path rehydration patch: PASS",
    "AST safety check: PASS",
]
missing = [m for m in required_patch_markers if m not in patch]
if missing:
    raise RuntimeError(f"Pinned repair patch missing marker(s): {missing}")

start_marker = "# Static safety assertions before execution.\n"
end_marker = 'print("Pinned Repair-7 v1.7 CPU patch verification: PASS")\n'
start = code.find(start_marker)
end = code.find(end_marker, start)
if start < 0 or end < 0 or end <= start:
    raise RuntimeError(
        f"Could not locate v1.7 safety block: start={start}, end={end}"
    )

code = code[:start] + patch.rstrip() + "\n\n" + code[end:]
code = code.replace(
    "DFU Repair-7 v1.7 CPU ONLY",
    "DFU Repair-7 CPU FINAL PLAIN-EVIDENCE",
    1,
)
code = code.replace(
    "Pinned Repair-7 v1.7 CPU patch verification: PASS",
    "Pinned Repair-7 CPU FINAL verification: PASS",
    1,
)

compile(code, "DFU_Repair7_CPU_FINAL_PLAIN_EVIDENCE.py", "exec")

print("DFU Repair-7 CPU FINAL notebook: PASS")
print("Evidence transport: PLAIN CSV")
print("Embedded hex/Base64 recovery: REMOVED")
print("CPU mode: ON")
print("38 good trials: READ-ONLY")
print("Authorized training: EXACT original 7")
print("Dataset split source: EXISTING locked split ONLY")

exec(
    compile(code, "DFU_Repair7_CPU_FINAL_PLAIN_EVIDENCE.py", "exec"),
    globals(),
)
